# Materialize Validated Gait Windows

This notebook converts the accepted preprocessing contract into a model-ready raw window store. It uses 5-second windows at 100 Hz with the common 18-channel order, participant-level split keys, Voisard annotated walking bounds, and the conservative Felius walking-candidate rule.

Two malformed Felius trials with header-only foot files are excluded and recorded in the audit. The output stores common-unit signals as float32; fold-specific normalization remains a training-time operation to prevent leakage.

In [1]:
import json
import sys
from fractions import Fraction
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import correlate, resample_poly

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

manifest = pd.read_csv(PROJECT_ROOT / 'data' / 'interim' / 'ml_readiness_manifest.csv')
participant_splits = pd.read_csv(PROJECT_ROOT / 'data' / 'interim' / 'participant_splits.csv')
TARGET_FS_HZ = 100.0
WINDOW_SECONDS = 5.0
WINDOW_SAMPLES = int(TARGET_FS_HZ * WINDOW_SECONDS)
TRAIN_HOP_SAMPLES = WINDOW_SAMPLES // 2
EVAL_HOP_SAMPLES = WINDOW_SAMPLES
CHANNEL_ORDER = [
    'LB_acc_x', 'LB_acc_y', 'LB_acc_z', 'LB_gyr_x', 'LB_gyr_y', 'LB_gyr_z',
    'LF_acc_x', 'LF_acc_y', 'LF_acc_z', 'LF_gyr_x', 'LF_gyr_y', 'LF_gyr_z',
    'RF_acc_x', 'RF_acc_y', 'RF_acc_z', 'RF_gyr_x', 'RF_gyr_y', 'RF_gyr_z',
]
ACCELERATION_MS2_TO_G = 1.0 / 9.80665
RAD_TO_DEG = 180.0 / np.pi
ACTIVITY_THRESHOLD_G = 0.02
PERIODICITY_THRESHOLD = 0.25
BAD_FELIUS_TRIALS = {
    'S8099Z_Stroke_Longitudinal_T2_Ja',
    'S9848Z_Stroke_Longitudinal_T2_Nee',
}
print('Manifest rows:', len(manifest))
print('Window shape:', (WINDOW_SAMPLES, len(CHANNEL_ORDER)))


Manifest rows: 865
Window shape: (500, 18)


In [2]:
def resample_signal(signal, source_fs):
    if float(source_fs) == TARGET_FS_HZ:
        return signal
    ratio = Fraction(float(TARGET_FS_HZ) / float(source_fs)).limit_denominator(1000)
    return resample_poly(signal, ratio.numerator, ratio.denominator, axis=0)


def voisard_bounds(meta):
    uturn_start, uturn_end = meta['uturnBoundaries']
    events = []
    for event_name in ['leftGaitEvents', 'rightGaitEvents']:
        events.extend(meta.get(event_name) or [])
    pre = [event for event in events if event[1] < uturn_start]
    post = [event for event in events if event[0] > uturn_end]
    bounds = []
    if pre:
        bounds.append((min(event[0] for event in pre), max(event[1] for event in pre)))
    if post:
        bounds.append((min(event[0] for event in post), max(event[1] for event in post)))
    return sorted(bounds)


def load_voisard(row):
    trial_dir = PROJECT_ROOT / row['trial_directory']
    meta = json.loads((trial_dir / f"{row['trial_id']}_meta.json").read_text(encoding='utf-8'))
    frames = []
    for sensor in ['LB', 'LF', 'RF']:
        frame = pd.read_csv(trial_dir / f"{row['trial_id']}_raw_data_{sensor}.txt", sep='\t')
        frame = frame[['Acc_X', 'Acc_Y', 'Acc_Z', 'Gyr_X', 'Gyr_Y', 'Gyr_Z']].astype(float)
        values = frame.to_numpy()
        values[:, :3] *= ACCELERATION_MS2_TO_G
        values[:, 3:] *= RAD_TO_DEG
        frames.append(values)
    n_samples = min(len(frame) for frame in frames)
    signal = np.concatenate([frame[:n_samples] for frame in frames], axis=1)
    return resample_signal(signal, float(meta['freq'])), voisard_bounds(meta), 'voisard_event_bounds'


def load_felius(row):
    paths = {}
    for raw_path in str(row['raw_files']).split('|'):
        sensor = Path(raw_path).name.rsplit('_', 1)[-1].replace('.csv', '')
        paths[sensor] = PROJECT_ROOT / raw_path
    frames = []
    for sensor in ['lowback', 'leftfoot', 'rightfoot']:
        frame = pd.read_csv(paths[sensor])
        frame = frame[['ax', 'ay', 'az', 'gx', 'gy', 'gz']].astype(float)
        frames.append(frame.to_numpy())
    n_samples = min(len(frame) for frame in frames)
    signal = np.concatenate([frame[:n_samples] for frame in frames], axis=1)
    return signal, [(0, n_samples)], 'felius_signal_candidate'


def accel_magnitude(signal, placement_start):
    return np.linalg.norm(signal[:, placement_start:placement_start + 3], axis=1)


def periodicity_strength(signal_1d):
    x = np.asarray(signal_1d, dtype=float)
    min_lag = int(0.4 * TARGET_FS_HZ)
    max_lag = int(2.0 * TARGET_FS_HZ)
    if len(x) < 2 * max_lag or not np.isfinite(x).all():
        return np.nan
    x = x - np.mean(x)
    if np.std(x) == 0:
        return np.nan
    ac = correlate(x, x, mode='full', method='fft')[len(x) - 1:]
    if ac[0] <= 0:
        return np.nan
    search = ac[min_lag:max_lag] / ac[0]
    peak_offset = int(np.argmax(search))
    if peak_offset <= 1 or peak_offset >= len(search) - 2:
        return np.nan
    return float(search[peak_offset])


def felius_candidate(signal, start):
    end = start + WINDOW_SAMPLES
    left = accel_magnitude(signal, 6)[start:end]
    right = accel_magnitude(signal, 12)[start:end]
    left_activity = float(np.std(left))
    right_activity = float(np.std(right))
    foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))
    return (
        left_activity >= ACTIVITY_THRESHOLD_G
        and right_activity >= ACTIVITY_THRESHOLD_G
        and foot_periodicity >= PERIODICITY_THRESHOLD
    )


## Build the window index

Voisard windows come from annotated walking bounds. Felius windows are selected from 5-second candidates with a 2.5-second hop. The index is built before allocating the array so the storage shape is known exactly.

In [3]:
window_specs = []
trial_audit = []
for _, row in manifest[manifest['dataset_id'].isin(['voisard_2025', 'felius_2024'])].iterrows():
    trial_id = row['trial_id']
    if row['dataset_id'] == 'felius_2024' and trial_id in BAD_FELIUS_TRIALS:
        trial_audit.append({'trial_id': trial_id, 'dataset_id': row['dataset_id'], 'label': row['label'], 'n_samples': 0, 'windows': 0, 'status': 'excluded_malformed_header_only_sensor'})
        continue

    if row['dataset_id'] == 'voisard_2025':
        signal, bounds, method = load_voisard(row)
        starts = [start for bound_start, bound_end in bounds for start in range(int(bound_start), int(bound_end) - WINDOW_SAMPLES + 1, TRAIN_HOP_SAMPLES)]
        selected = [(start, 'voisard_annotated_walking') for start in starts]
    else:
        signal, bounds, method = load_felius(row)
        if len(signal) < WINDOW_SAMPLES:
            trial_audit.append({'trial_id': trial_id, 'dataset_id': row['dataset_id'], 'label': row['label'], 'n_samples': len(signal), 'windows': 0, 'status': 'excluded_short_or_empty'})
            continue
        selected = [(start, 'felius_periodic_walking_candidate') for start in range(0, len(signal) - WINDOW_SAMPLES + 1, TRAIN_HOP_SAMPLES) if felius_candidate(signal, start)]

    trial_audit.append({'trial_id': trial_id, 'dataset_id': row['dataset_id'], 'label': row['label'], 'n_samples': len(signal), 'windows': len(selected), 'status': 'included' if selected else 'no_valid_windows'})
    participant_key = f"{row['dataset_id']}:{row['subject']}"
    for start, selection_rule in selected:
        window_specs.append({
            'dataset_id': row['dataset_id'],
            'participant_key': participant_key,
            'trial_id': trial_id,
            'label': row['label'],
            'start_sample': int(start),
            'end_sample': int(start + WINDOW_SAMPLES),
            'window_seconds': WINDOW_SECONDS,
            'selection_rule': selection_rule,
        })

window_metadata = pd.DataFrame(window_specs)
trial_audit = pd.DataFrame(trial_audit)
window_metadata.insert(0, 'window_id', np.arange(len(window_metadata), dtype=np.int64))
print('Window count:', len(window_metadata))
print(window_metadata.groupby(['dataset_id', 'label']).size())
print()
print('Trial status:')
print(trial_audit.groupby(['dataset_id', 'status']).size())


C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))


C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))
C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))


C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))


C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))


C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))
C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))
C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))
C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))
C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), period

...

C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))


C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))


C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))
C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))
C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))
C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))
C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), period

Window count: 18511
dataset_id    label  
felius_2024   healthy     2921
              stroke     13445
voisard_2025  healthy     1039
              stroke      1106
dtype: int64

Trial status:
dataset_id    status                               
felius_2024   excluded_malformed_header_only_sensor      2
              included                                 368
              no_valid_windows                           7
voisard_2025  included                                 482
              no_valid_windows                           6
dtype: int64


C:\Users\frank\AppData\Local\Temp\ipykernel_6424\2114724846.py:83: RuntimeWarning: All-NaN axis encountered
  foot_periodicity = float(np.nanmax([periodicity_strength(left), periodicity_strength(right)]))


In [4]:
processed = PROJECT_ROOT / 'data' / 'processed'
processed.mkdir(parents=True, exist_ok=True)
metadata_path = processed / 'validated_window_metadata.csv'
audit_path = processed / 'validated_trial_materialization_audit.csv'
array_path = processed / 'validated_gait_windows_float32.npy'
window_metadata.to_csv(metadata_path, index=False)
trial_audit.to_csv(audit_path, index=False)

shape = (len(window_metadata), WINDOW_SAMPLES, len(CHANNEL_ORDER))
window_array = np.lib.format.open_memmap(array_path, mode='w+', dtype='float32', shape=shape)
for trial_id, specs in window_metadata.groupby('trial_id', sort=False):
    row = manifest[manifest['trial_id'] == trial_id].iloc[0]
    signal, _, _ = load_voisard(row) if row['dataset_id'] == 'voisard_2025' else load_felius(row)
    for spec in specs.itertuples(index=False):
        window_array[spec.window_id] = signal[spec.start_sample:spec.end_sample].astype(np.float32)
window_array.flush()

print('Array:', array_path)
print('Array shape:', shape)
print('Array size (MB):', round(array_path.stat().st_size / 1024**2, 1))
print('Metadata:', metadata_path)
print('Trial audit:', audit_path)


Array: C:\Users\frank\Documents\MR-ICT Review Paper\data\processed\validated_gait_windows_float32.npy
Array shape: (18511, 500, 18)
Array size (MB): 635.5
Metadata: C:\Users\frank\Documents\MR-ICT Review Paper\data\processed\validated_window_metadata.csv
Trial audit: C:\Users\frank\Documents\MR-ICT Review Paper\data\processed\validated_trial_materialization_audit.csv


In [5]:
split_check = window_metadata.merge(participant_splits, on='participant_key', how='left')
assert split_check['fold'].notna().all()
assert np.isfinite(np.asarray(window_array)).all()
print('Split join: OK')
print('Finite array values: OK')
print('Window counts by fold role and label:')
print(split_check.groupby(['fold', 'role', 'label']).size())


Split join: OK
Finite array values: OK
Window counts by fold role and label:
fold  role        label  
0     training    healthy     3021
                  stroke     11357
      validation  healthy      939
                  stroke      3194
1     training    healthy     3095
                  stroke     11632
      validation  healthy      865
                  stroke      2919
2     training    healthy     3240
                  stroke     11501
      validation  healthy      720
                  stroke      3050
3     training    healthy     3026
                  stroke     11488
      validation  healthy      934
                  stroke      3063
4     training    healthy     3458
                  stroke     12226
      validation  healthy      502
                  stroke      2325
dtype: int64


## Output contract for model training

Use validated_gait_windows_float32.npy with memory mapping, join validated_window_metadata.csv to participant_splits.csv, and calculate channel normalization statistics using only the training participants inside each fold. Do not randomly split rows because overlapping windows from one participant would otherwise leak into validation.